# Build Your Own StudyBot
Run cells top to bottom (**Runtime → Run all**). Before starting: add your Gemini key in Colab secrets (key icon on the left) as `GEMINI_API_KEY`, and upload `Campbell_Chapters_3-5.pdf` and `test_set.json` in the folder panel.

## Section 0: setup (Siddarth)

In [ ]:
!pip install -q -U google-genai pypdf sentence-transformers numpy gradio==5.49.1

from google import genai
from google.colab import userdata
import time

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
                      http_options={"timeout": 30000})   # give up after 30 seconds

In [ ]:
def plain_llm(question, retries=3):
    for attempt in range(retries):
        try:
            r = client.models.generate_content(
                model="gemini-flash-lite-latest", contents=question,
                config={"max_output_tokens": 300})
            return r.text or ""
        except Exception as e:
            print("retrying:", str(e)[:100])
            time.sleep(5 * (attempt + 1))
    return ""

print(plain_llm("Say hi in one word."))   # quick check that your API key works

In [ ]:
# Choose your notes PDF
SAMPLE = "Campbell_Chapters_3-5.pdf"
USE_MY_OWN_PDF = False          # students: change to True to upload your own notes

pdf_path = SAMPLE
if USE_MY_OWN_PDF:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        pdf_path = next(iter(uploaded))
print("Using:", pdf_path)

## Step 1: load the PDF (Siddarth)

In [ ]:
from pypdf import PdfReader
import re

def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):                      # TODO-STUDENT
        text = re.sub(r"\s+", " ", page.extract_text() or "").strip()    # TODO-STUDENT
        text = re.sub(r"# 152561 Cust:.*?Publishing Services", "", text)        # remove printer header
        text = re.sub(r"\S*\.indd \d+ \d+/\d+/\d+ \d+:\d+ [AP]M", "", text)    # remove print footer
        text = text.strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

pages = load_pdf(pdf_path)
if len(pages) == 0:
    print("WARNING: no text found (is it a scanned PDF?). Switching to the sample PDF.")
    pdf_path = SAMPLE
    pages = load_pdf(pdf_path)
print(len(pages), "pages")
print(pages[0]["text"][:300])

## Step 2: chunking (Siddant)

In [ ]:
def chunk(pages, size=500, overlap=50):
    assert overlap < size
    chunks = []
    for p in pages:
        text = p["text"]
        for start in range(0, len(text), size - overlap):   # TODO-STUDENT
            piece = text[start:start + size]                 # TODO-STUDENT
            chunks.append({"id": len(chunks), "page": p["page"], "text": piece})
    return chunks

chunks = chunk(pages)
print(len(chunks), "chunks")
print(chunks[0])

In [ ]:
# Demo: the end of chunk 0 repeats at the start of chunk 1 (that's the overlap)
print("END OF CHUNK 0:  ...", chunks[0]["text"][-80:])
print("START OF CHUNK 1:", chunks[1]["text"][:80], "...")
print()
for size in (100, 2000):
    print(f"size {size}: {len(chunk(pages, size=size, overlap=50))} chunks")

## Step 3: embeddings (Madhur)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return model.encode(texts, normalize_embeddings=True)   # TODO-STUDENT

vectors = embed([c["text"] for c in chunks])
print(vectors.shape)

In [ ]:
# Demo: similar meaning = high score, different meaning = lower score
s = embed(["I failed the exam", "I flunked the test", "I passed the exam"])
print("failed vs flunked:", round(float(s[0] @ s[1]), 2))   # TODO-STUDENT
print("failed vs passed: ", round(float(s[0] @ s[2]), 2))

## Step 4: search (Abhiman)

In [ ]:
def search(question, chunks, vectors, k=3):
    """Return the top-k chunks for `question`, each with a "score", best first."""
    # 1. Turn the question into a vector. Pass a LIST, then take row 0
    q = model.encode([question], normalize_embeddings=True)[0]
    # 2. Similarity of the question to every chunk
    scores = vectors @ q  # TODO-STUDENT
    # 3. Indices of the k highest scores, best first
    top = np.argsort(scores)[::-1][:k]  # TODO-STUDENT
    # 4. Return copies of those chunks with their score attached
    return [{**chunks[i], "score": round(float(scores[i]), 3)} for i in top]

for r in search("What is the main topic of these notes?", chunks, vectors):
    print(r["page"], r["score"], r["text"][:100])

## Step 5: answer with an LLM (Adarsh)

In [ ]:
def answer(question, top_chunks):
    context = "\n\n".join(f"[{n}] (p.{c['page']}) {c['text']}"
                          for n, c in enumerate(top_chunks, start=1))     # TODO-STUDENT
    prompt = ("You are a study helper. Answer ONLY from the context below. "
              "If the answer isn't in the context, say exactly: That's not in your notes.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}")
    return plain_llm(prompt)                                              # TODO-STUDENT

# Same question, without and with the notes
q = "According to the notes, how many Calories are in a cheeseburger?"
print("WITHOUT NOTES:\n", plain_llm(q), "\n")
print("WITH NOTES:\n", answer(q, search(q, chunks, vectors, k=3)))

## Step 6: citations + "I don't know" (Shagun)

In [ ]:
def ask(question, k=3, min_score=0.5):
    top = search(question, chunks, vectors, k=k)
    # Tune min_score if you use your own PDF: the right value depends on the notes
    if top[0]["score"] < min_score:                                       # TODO-STUDENT
        return "That's not in your notes."
    reply = answer(question, top)
    if "not in your notes" in reply.lower():
        return reply                                   # don't add sources to a refusal
    pages_used = sorted({c["page"] for c in top})                         # TODO-STUDENT
    return reply + "\n\nSources: " + ", ".join(f"p. {p}" for p in pages_used)

# 3 demo questions: answered, split across pages, off-topic
for q in ["According to the notes, how many Calories are in a cheeseburger?",
          "What molecule usually provides the energy for active transport, and what process regenerates a cell's supply of it?",
          "How do I bake a chocolate cake?"]:
    print("Q:", q)
    print("A:", ask(q), "\n")

## Step 7: web app (Saksham)
Your `ask()` plugs straight into the Study Bot interface: no files to export. The cell below fetches the interface and launches it on top of your bot.

In [ ]:
import os, subprocess, sys

# The polished interface lives in the Study-Bot repo. Clone it once.
REPO_URL = "https://github.com/sakshamdev/study-bot.git"   # TODO-ORGANIZER: put your repo URL here
if not os.path.exists("Study-Bot"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "Study-Bot"])

def my_ask(question):
    # The interface only needs: question in, answer text out.
    return ask(question, k=5, min_score=0.5)                              # TODO-STUDENT

try:
    sys.path.insert(0, "Study-Bot")
    from app import launch_with
    launch_with(my_ask)          # polished interface + YOUR bot, with a share link
except Exception as e:
    print("Polished interface unavailable:", str(e)[:150], "\nUsing the simple interface instead.")
    import gradio as gr
    gr.ChatInterface(
        fn=lambda message, history: my_ask(message),
        title="My StudyBot",
        examples=["How many Calories are in a cheeseburger?",
                  "What is hydrolysis?",
                  "What does the smooth ER do in liver cells?"],
    ).launch(share=True, debug=False)

## Step 8: test and tune (Shivani)

In [ ]:
import json
test_set = json.load(open("test_set.json"))
print(len(test_set), "questions")

# retrieval check: do the top-3 chunks come from the expected pages?
for item in test_set:
    top = search(item["question"], chunks, vectors, k=3)
    got = [t["page"] for t in top]
    print(item["type"], "| expected", item["pages"], "| got", got,
          "| best score", round(top[0]["score"], 2))

In [ ]:
import pandas as pd

IDK = ["not in your notes", "don't know", "do not know", "not mentioned", "cannot find"]

def is_correct(item, reply):
    r = reply.lower().split("sources:")[0]         # ignore the page list
    if item["type"] == "absent":
        return any(p in r for p in IDK)
    return all(w.lower() in r for w in item["keywords"])   # TODO-STUDENT

def evaluate(test_set, settings):
    rows = []
    for item in test_set:
        q = item["question"]
        print("asking:", q[:50])
        plain = plain_llm(q)
        rag_reply = ask(q, settings["k"], settings["min_score"])
        if plain == "" or rag_reply == "":
            print("WARNING: API call failed for:", q[:50], "- this row is not reliable")
        rows.append({"question": q, "type": item["type"],
                     "plain_answer": plain, "rag_answer": rag_reply,
                     "plain_correct": is_correct(item, plain),
                     "rag_correct": is_correct(item, rag_reply)})
        time.sleep(4)                      # stay under the free-tier rate limit
    df = pd.DataFrame(rows)
    print("plain:", df.plain_correct.mean(), "| RAG:", df.rag_correct.mean())
    return df

In [ ]:
df = evaluate(test_set, {"k": 3, "min_score": 0.5})
df[["question", "type", "plain_correct", "rag_correct"]]

In [ ]:
assert isinstance(df, pd.DataFrame)
assert len(df) == len(test_set)
assert {"plain_correct", "rag_correct"} <= set(df.columns)
print("Contract check passed")

In [ ]:
# Why did question 6 fail? Compare what the bot found with k=3 vs k=5
q = test_set[6]
print("QUESTION:", q["question"])
print("NEEDS PAGES:", q["pages"])
for k in (3, 5):
    top = search(q["question"], chunks, vectors, k=k)
    print(f"k={k} found pages:", [t["page"] for t in top])

# Now ask again with k=5
reply = ask(q["question"], k=5, min_score=0.5)
print("\nANSWER WITH k=5:", reply)
print("CORRECT?", is_correct(q, reply))

In [ ]:
_cache = {500: (chunks, vectors)}   # size 500 is already embedded, reuse it
def get_index(size):                # chunk + embed once per size, then reuse
    if size not in _cache:
        ch = chunk(pages, size=size, overlap=50)
        _cache[size] = (ch, embed([c["text"] for c in ch]))
        print(f"size {size}: {len(ch)} chunks")
    return _cache[size]

def sweep(test_set, sizes=(200, 500, 1000), ks=(1, 3, 5)):
    rows = []
    for size in sizes:
        ch, vec = get_index(size)
        for k in ks:
            hits, n = 0, 0
            for item in test_set:
                if item["type"] == "absent":
                    continue
                top = search(item["question"], ch, vec, k=k)
                ok = set(item["pages"]) <= {t["page"] for t in top}   # TODO-STUDENT
                hits += ok
                n += 1
            rows.append({"size": size, "k": k, "score": round(hits / n, 2)})
    table = pd.DataFrame(rows).pivot(index="size", columns="k", values="score")
    print(table)
    return table

table = sweep(test_set)

In [ ]:
# Try it on YOUR notes: ask something only your notes would know
my_question = "Type a question about your notes here"   # TODO-STUDENT

print("PLAIN CHATBOT:\n", plain_llm(my_question), "\n")
print("YOUR STUDYBOT:\n", ask(my_question, k=5, min_score=0.5))

## Step 9 (optional): run on your own computer
Step 7 already gives you a shareable web app from Colab. To run the same interface locally instead, run the next cell to download `my_backend.py`, then follow the repo README (`STUDYBOT_BACKEND=my_backend`).

In [ ]:
import inspect
from google.colab import files

HEADER = """\"\"\"My StudyBot backend, exported from the workshop notebook.
Needs env vars GEMINI_API_KEY and (optionally) NOTES_PDF (default Campbell_Chapters_3-5.pdf).
\"\"\"
import os, re, time
import numpy as np
from google import genai
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

"""

FOOTER = """

# ---- build the index once, when the module is imported ----
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"], http_options={"timeout": 30000})
model = SentenceTransformer("all-MiniLM-L6-v2")
pages = load_pdf(os.getenv("NOTES_PDF", "Campbell_Chapters_3-5.pdf"))
chunks = chunk(pages)
vectors = embed([c["text"] for c in chunks])

# ---- the web app wants failures raised, not hidden as empty answers ----
_plain_llm = plain_llm
def plain_llm(question, retries=3):
    reply = _plain_llm(question, retries)
    if not reply:
        raise RuntimeError("The language model did not respond")
    return reply
"""

funcs = [load_pdf, chunk, embed, plain_llm, search, answer, ask]
code = HEADER + "\n\n".join(inspect.getsource(f) for f in funcs) + FOOTER
open("my_backend.py", "w", encoding="utf-8").write(code)
print("Wrote my_backend.py (%d lines)" % len(code.splitlines()))
files.download("my_backend.py")

## What to build next
- Ask your bot to make quiz questions from your notes
- Upload more than one PDF (a whole semester)
- Remember earlier questions in the chat
- Write 10 test questions for your own notes and run the sweep on them